***Alexandre Mathias DONNAT, Sr - Télécom Paris***

In [ ]:
"""Relation Classification Lab

### === Purpose ===

The goal of this lab is to perform relation classification on a text where NER and Disambiguation were performed. For example, given a Wikipedia article:

    <Elvis_Presley>
    <Elvis_Presley> was an <United_States_of_America> singer and actor, married to <Priscilla_Presley>.

the goal is to predict the relation between the title entity and the others:

    <Elvis_Presley><nationality><United_States_of_America>
    <Elvis_Presley><spouse><Priscilla_Presley>

We will use a Language Model for this task, and make use of Constrained Decoding in order to make the predictions.

=== Provided Data ===

We provide
1. A preprocessed version of Wikipedia, wikipedia-ner.txt, which contains articles about disambiguated entities, whose content also went through NERC and Disambiguation.
2. A gold standard for the task, student-gold-standard.tsv, which contains triples <subject_entity> <object_entity> <relation>, that we will use to evaluate our method
3. a template for our code, relation_classification.py

### === Task ===
We will have two tasks in this lab.
The first will be to complete the function construct_trie, so that it constructs a trie for the (tokenized) list of relations given as input.
Our second task is to complete the function classify_relation in this file.
It receives as input (1) the title entity (subject), (2) the article content, (3) a trie.
It outputs a list of relations between the title entity and all the other disambiguated entities in the article content. It uses Language Models and Constrained Decoding.

### === Working with Colab ===
We need to save a local copy of the notebook to our own google drive.
Connect to an execution environment using a GPU (this should be automatic, but be aware of this !). Upload the local files directly to the colab, and you can run everything !

Don't forget to download the results file at the end.

Install necessary modules with:
"""
!pip install -q transformers
!pip install -q sentencepiece
!pip install -q accelerate

In [2]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
from collections import defaultdict
import time
from tqdm import tqdm
import re
from typing import Dict, List

# Loads a T5 LLM
torch.cuda.empty_cache()
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large", device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [3]:

class WikipediaArticle:
  """ Represents a Wikipedia article. Do not modify. """
  def __init__(self, title, content):
    self.title_entity=title
    self.content=content

def wikipediaArticles(file):
  """ Yields the Wikipedia articles from a file. Do not modify. """
  article=[]
  title=None
  with open(file, "rt", encoding="utf=8") as inputFile:
    for line in inputFile:
      line=line.rstrip()
      if not title:
        title=line
        continue
      if not len(line) and title and len(article):
        yield WikipediaArticle(title, article[0])
        title=None
        article=[]
        continue
      article+=[line]

def clean(yagoEntity):
    """ Removes prefixes etc."""
    if not yagoEntity:
        return ""
    if yagoEntity.startswith('"'):
      return yagoEntity[1:-1]
    yagoEntity=yagoEntity[yagoEntity.find(':')+1:]
    return '<'+yagoEntity+'>'

In [ ]:
def run_evaluation():
  """Evaluation script, do not modify (unless we want to remove some prints).
  We use the f-05 measure, which gives more importance to precision: classifying entities correctly is more valued than finding all entities.
  """
  with open("student-gold-standard.tsv", "r", encoding="utf-8") as f:
    lines = f.readlines()
  gold_standard_dict = defaultdict(dict)
  for line in lines:
    title_entity, entity_id, relation = tuple(line.replace("\n","").split("\t"))
    gold_standard_dict[title_entity][entity_id] = relation
  gold_standard_dict = dict(gold_standard_dict)

  with open("results.tsv", "r", encoding="utf-8") as f:
    lines = f.readlines()
  predictions_dict = defaultdict(dict)
  for line in lines:
    title_entity, entity_id, relation = tuple(line.replace("\n","").split("\t"))
    predictions_dict[title_entity][entity_id] = relation

  true_pos = 0
  false_pos = 0
  false_neg = 0

  for title_entity in predictions_dict:
    for entity_id in predictions_dict[title_entity]:
      try:
        gold_yago_relation = gold_standard_dict[title_entity][entity_id]
      except KeyError: #should not happen
        continue
      if predictions_dict[title_entity][entity_id] == gold_yago_relation:
        true_pos += 1
      else:
        false_pos += 1
        if false_pos < 100:
          print("We classified the relation between", title_entity + " and " + entity_id, "wrong.", "Expected output: ", gold_yago_relation, ",given:", predictions_dict[title_entity][entity_id])

  for gold_title in gold_standard_dict:
    for entity_id in gold_standard_dict[gold_title]:
      try:
        predict_relation = predictions_dict[gold_title][entity_id]
      except KeyError:
        false_neg += 1
        if false_neg < 100:
          print("We did not classify the relation between", gold_title + " and " + entity_id +".")

  if true_pos + false_pos != 0:
    precision = float(true_pos) / (true_pos + false_pos)
  else:
    precision = 0.0

  if true_pos + false_neg != 0:
    recall = float(true_pos) / (true_pos + false_neg)
  else:
    recall = 0.0

  beta = 0.5

  if precision + recall != 0.0:
    f05 = (1 + beta * beta) * precision * recall / (beta * beta * precision + recall)
  else:
    f05 = 0.0

  print()
  print("Scores (scaled from 0 to 100)")
  print("Precision", precision*100)
  print("Recall", recall*100)
  print("F-0.5 Score", f05*100)

def get_all_relations(file):
  with open(file, "r", encoding="utf-8") as f:
    lines = f.readlines()
  relations = set()
  for line in lines:
    title_entity, entity_id, relation = tuple(line.replace("\n","").split("\t"))
    relations.add(relation)
  return relations

def prefix_allowed_tokens_fn(input_ids, trie, prompt_len: int):
  '''
  The function that handles constrained decoding.
  For the current generated text, returns the following allowed tokens. If nothing is allowed, return EOS token (ends the generation).
  This function is called at every generation step (every time a token is generated)
  '''
  model_output = input_ids.tolist()
  allowed_tokens = trie.get(model_output)
  if not allowed_tokens:
    return [tokenizer.eos_token_id]
  return allowed_tokens

In [5]:
class Trie(object):
    def __init__(self, sequences: List[List[int]] = []):
        self.trie_dict = {}
        if sequences:
            for sequence in sequences:
                Trie._add_to_trie(sequence, self.trie_dict)

    def add(self, sequence: List[int]):
        Trie._add_to_trie(sequence, self.trie_dict)

    def get(self, prefix_sequence: List[int]):
        return Trie._get_from_trie(prefix_sequence, self.trie_dict)

    @staticmethod
    def _add_to_trie(sequence: List[int], trie_dict: Dict):
        if sequence:
            if sequence[0] not in trie_dict:
                trie_dict[sequence[0]] = {}
            Trie._add_to_trie(sequence[1:], trie_dict[sequence[0]])

    @staticmethod
    def _get_from_trie(prefix_sequence: List[int], trie_dict: Dict):
        if len(prefix_sequence) == 0:
            output = list(trie_dict.keys())
            return output
        elif prefix_sequence[0] in trie_dict:
            return Trie._get_from_trie(prefix_sequence[1:],trie_dict[prefix_sequence[0]])
        else:
            return []

    def __iter__(self):
        def _traverse(prefix_sequence, trie_dict):
            if trie_dict:
                for next_token in trie_dict:
                    yield from _traverse(prefix_sequence + [next_token], trie_dict[next_token])
            else:
                yield prefix_sequence

        return _traverse([], self.trie_dict)

    def __getitem__(self, value):
        return self.get(value)

In [6]:
def construct_trie(relations: List[str], tokenizer):
    """
    Builds a Trie over tokenized relation labels.

    In T5 generation, decoding starts with the pad token.
    Therefore each valid generated sequence must start with tokenizer.pad_token_id.
    """
    trie = Trie()

    start_token_id = tokenizer.pad_token_id
    eos_token_id = tokenizer.eos_token_id

    for relation in sorted(set(relations)):
        relation = relation.strip()
        if not relation:
            continue

        relation_tokens = tokenizer.encode(
            relation,
            add_special_tokens=False
        )

        trie.add([start_token_id] + relation_tokens + [eos_token_id])

    return trie

In [7]:
def run_model(prompt, trie):
  """
  Runs the language model using constrained decoding
  Input:
  - Prompt (str)
  - prefix_allowed_tokens_fn
  """
  device = "cuda" if torch.cuda.is_available() else "cpu"
  input_text = prompt
  inputs = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

  prompt_len = inputs.shape[-1]

  outputs = model.generate(inputs, max_new_tokens=20, do_sample=False, num_beams=5, temperature=None, top_p=None, pad_token_id=tokenizer.eos_token_id,
        prefix_allowed_tokens_fn=lambda _, input_ids: prefix_allowed_tokens_fn(input_ids, trie, prompt_len))
  return(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [8]:
def classify_relations(title_entity, page_content, trie):
    """
    Predicts relations between the title entity and the other entities appearing in the article.
    Returns a list of triples: (title_entity, object_entity, relation)
    """

    entities = re.findall(r"<[^<>]+>", page_content)

    object_entities = []
    seen = set()

    for entity in entities:
        if entity != title_entity and entity not in seen:
            seen.add(entity)
            object_entities.append(entity)

    results = []

    def get_context(text, obj, window=700):
        """
        Keeps a short local context around the object entity.
        This avoids sending the full article to the model.
        """
        match = re.search(re.escape(obj), text)

        if match is None:
            snippet = text[:1400]
        else:
            start = max(0, match.start() - window)
            end = min(len(text), match.end() + window)
            snippet = text[start:end]

        snippet = re.sub(r"\s+", " ", snippet)
        return snippet

    for obj in object_entities:
        context = get_context(page_content, obj)

        prompt = (
            "Classify the semantic relation expressed in the text. "
            "The subject is the Wikipedia title entity. "
            "Return only the relation label.\n\n"
            f"Subject: {title_entity}\n"
            f"Object: {obj}\n"
            f"Text: {context}\n\n"
            "Relation:"
        )

        relation = run_model(prompt, trie).strip()

        if relation:
            results.append((title_entity, obj, relation))

    return results

In [9]:
def run():
  relations = get_all_relations("student-gold-standard.tsv")
  trie = construct_trie(relations, tokenizer)

  with open("results.tsv", 'w', encoding="utf-8") as output:
    start = time.time()

    for i, page in enumerate(wikipediaArticles("wikipedia-ner.txt")):
      print("Processing", page.title_entity, i)

      result = classify_relations(page.title_entity, page.content, trie)

      if result is not None:
        for subj, obj, rel in result:
          output.write(subj + "\t" + obj + "\t" + rel + "\n")

    end = time.time()

  print("done")
  print("execution time:", end - start)
  print("number of articles:", i)

In [10]:
run()

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Processing <Ashok_Kumar__u0028_Indian_politician_u0029_> 0
Processing <Ashok_Kumar__u0028_golfer_u0029_> 1
Processing <Ashok_Kumar_Dogra> 2
Processing <Ashok_Kumar> 3
Processing <Ashok_Kumar_Garg> 4
Processing <Ashok_Kumar__u0028_British_politician_u0029_> 5
Processing <Ashok_Kumar__u0028_politician_u0029_> 6
Processing <Ashok_Kumar__u0028_cinematographer_u0029_> 7
Processing <Mortal_Kombat__u0028_2011_video_game_u0029_> 8
Processing <Mortal_Kombat__u0028_1992_video_game_u0029_> 9
Processing <Mortal_Kombat__u0028_2021_film_u0029_> 10
Processing <Mortal_Kombat__u0028_1995_film_u0029_> 11
Processing <Epirus__u0028_ancient_state_u0029_> 12
Processing <Epirus> 13
Processing <Marble_Canyon__u0028_British_Columbia_u0029_> 14
Processing <Marble_Canyon__u0028_Canadian_Rockies_u0029_> 15
Processing <Marble_Canyon_u002C__Arizona> 16
Processing <Marble_Canyon> 17
Processing <Foxfire__u0028_1996_film_u0029_> 18
Processing <Foxfire__u0028_1987_film_u0029_> 19
Processing <Foxfire_u002C__North_Caroli

In [13]:
run_evaluation()

You classified the relation between <Ashok_Kumar__u0028_Indian_politician_u0029_> and <1954-10-27T00:00:00Z> wrong. Expected output:  <birthDate> ,given: <nationality>
You classified the relation between <Ashok_Kumar__u0028_Indian_politician_u0029_> and <Indian_National_Congress> wrong. Expected output:  <memberOf> ,given: <birthPlace>
You classified the relation between <Ashok_Kumar__u0028_Indian_politician_u0029_> and <Patna_Medical_College_and_Hospital> wrong. Expected output:  <alumniOf> ,given: <nationality>
You classified the relation between <Ashok_Kumar__u0028_golfer_u0029_> and <1983-07-20T00:00:00Z> wrong. Expected output:  <birthDate> ,given: <manufacturer>
You classified the relation between <Ashok_Kumar__u0028_golfer_u0029_> and <Bihar> wrong. Expected output:  <birthPlace> ,given: <nationality>
You classified the relation between <Ashok_Kumar_Dogra> and <1958-11-24T00:00:00Z> wrong. Expected output:  <birthDate> ,given: <deathPlace>
You classified the relation between <As

In [ ]:
def normalize_relation(rel):
    rel = rel.strip()

    rel = rel.replace("<pad>", "")
    rel = rel.replace("</s>", "")
    rel = rel.replace("<unk>", "<")
    rel = rel.strip()

    # "nationality>" -> "<nationality>"
    rel = rel.strip("<>")
    rel = rel.strip()

    if not rel:
        return None

    return f"<{rel}>"

# Fix results.tsv in place
fixed_lines = []

with open("results.tsv", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) != 3:
            continue

        subj, obj, rel = parts
        rel = normalize_relation(rel)

        if rel is not None:
            fixed_lines.append(subj + "\t" + obj + "\t" + rel + "\n")

with open("results.tsv", "w", encoding="utf-8") as f:
    f.writelines(fixed_lines)

print("Fixed", len(fixed_lines), "lines")

Fixed 799 lines
